# RQ4: Feature Importance and Interpretability
**Research Question:** Which input features contribute most to predicting Revenue_Generated, and what domain insights can be derived?

**Methods:** Random Forest feature importance + SHAP summary plot

In [3]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
import shap
import os

os.makedirs('figures', exist_ok=True)
os.makedirs('tables', exist_ok=True)

In [4]:
DATA_PATH = '/kaggle/input/datasets/karansridhar/karan-ue-ml-assignment-dataset/marketing_and_product_performance.csv'
df = pd.read_csv(DATA_PATH)
TARGET = 'Revenue_Generated'

id_cols = [c for c in df.columns if 'ID' in c or 'id' in c.lower()]
df = df.drop(columns=id_cols, errors='ignore').dropna(subset=[TARGET])

cat_cols = df.select_dtypes(include='object').columns
le = LabelEncoder()
for c in cat_cols:
    df[c] = le.fit_transform(df[c].fillna('Unknown').astype(str))

X = df.drop(columns=[TARGET])
y = df[TARGET]

imputer = SimpleImputer(strategy='median')
X_imp = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

X_train, X_test, y_train, y_test = train_test_split(X_imp, y, test_size=0.2, random_state=42)
print('Ready:', X_train.shape)

Ready: (8000, 11)


In [5]:
# Train Random Forest
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

importances = rf.feature_importances_
feat_df = pd.DataFrame({'Feature': X_imp.columns, 'Importance': importances})
feat_df = feat_df.sort_values('Importance', ascending=False).reset_index(drop=True)
feat_df['Rank'] = feat_df.index + 1
feat_df['Direction'] = feat_df['Feature'].apply(lambda f: 
    'Positive' if X_imp[f].corr(y) >= 0 else 'Negative')

top10 = feat_df.head(10)[['Rank','Feature','Importance','Direction']]
top10['Importance'] = top10['Importance'].round(4)
top10.to_csv('tables/RQ4_Table4_Feature_Importance.csv', index=False)
print(top10)

   Rank                            Feature  Importance Direction
0     1                       Bundle_Price      0.1264  Negative
1     2                             Budget      0.1256  Negative
2     3                             Clicks      0.1241  Negative
3     4                        Conversions      0.1194  Positive
4     5                                ROI      0.1167  Positive
5     6                         Units_Sold      0.1136  Positive
6     7                     Discount_Level      0.0998  Positive
7     8                Subscription_Length      0.0880  Positive
8     9  Customer_Satisfaction_Post_Refund      0.0322  Negative
9    10                    Common_Keywords      0.0297  Negative


In [6]:
# Figure 4a — Top-10 Feature Importance Bar Plot
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#1565C0' if d == 'Positive' else '#C62828' for d in top10['Direction']]
ax.barh(top10['Feature'][::-1], top10['Importance'][::-1], color=colors[::-1], edgecolor='white')
ax.set_xlabel('Feature Importance Score', fontsize=12)
ax.set_title('Figure 4: Top-10 Feature Importance (Random Forest)\nMarketing and Product Performance Dataset', fontsize=13, fontweight='bold')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#1565C0', label='Positive effect'),
                   Patch(facecolor='#C62828', label='Negative effect')]
ax.legend(handles=legend_elements, fontsize=10)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/RQ4_Figure4_Feature_Importance.pdf', bbox_inches='tight')
plt.show()
print('Figure 4 saved.')

Figure 4 saved.


In [ ]:
# Figure 4b — SHAP Summary Plot
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test.iloc[:500])  # sample for speed

fig, ax = plt.subplots(figsize=(10, 7))
shap.summary_plot(shap_values, X_test.iloc[:500], show=False, max_display=10)
plt.title('Figure 4b: SHAP Summary Plot — Revenue_Generated', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/RQ4_Figure4b_SHAP_Summary.pdf', bbox_inches='tight')
plt.show()
print('SHAP figure saved.')